# IN4080: obligatory assignment 4 - Part 2
 
The final mandatory assignment for IN4080 consists of two parts, which are distributed in two separate files. The first (a different notebook) is about the development of dialogue systems, and the second (this notebook) about machine translation. Both parts give a maximum of 10 points, and you are required to get at least 12/20 points to pass. Part 1 should be done on your local computer, as it relies on a speech interface that will not work on remote machines. For Part 2, using _Fox_ is preferable, at least for the fine-tuning task.

- We assume that you have read and are familiar with IFI’s requirements and guidelines for mandatory assignments, see [here](https://www.uio.no/english/studies/examinations/compulsory-activities/mn-ifi-mandatory.html) and [here](https://www.uio.no/english/studies/examinations/compulsory-activities/mn-ifi-guidelines.html).
- This is an individual assignment. You should not deliver joint submissions. 
- You may redeliver in Devilry before the deadline (__Friday, November 13 at 23:59__).
- Only the last delivery will be read! If you deliver more than one file, put them into a zip-archive. You don't have to include in your delivery the data files already provided for this assignment. 
- Submit two files and name your submissions _your\_username\_in4080\_mandatory\_4a_ / _your\_username\_in4080\_mandatory\_4b_

You should deliver a completed version of this Jupyter notebook, containing both your code and explanations about the steps you followed. We want to stress that simply submitting code is __not__ by itself sufficient to complete the assignment - we expect the notebook to also contain explanations of what you have implemented, along with motivations for the choices you made along the way. Preferably use whole sentences, and mathematical formulas if necessary. Explaining in your own words (using concepts we have covered through in the lectures) what you have implemented and reflecting on your solution is an important part of the learning process - take it seriously!

Regarding the use of LLMs (Claude, ChatGPT or similar): you are allowed to use them as 'sparring partner', for instance to clarify something you have not understood. However, you are __not__ allowed to use them to generate solutions (either in part or in full) to the assignment tasks. 

## Part 2 : Machine translation

In this part, we evaluate a pre-trained machine translation model on data from the Lord of the Rings movies and fine-tune it to improve the translation quality.

### Data

We provide you with two files, `lotr.detok.de` and `lotr.detok.en`, containing German and English movie subtitles. These two files constitute a so-called _parallel corpus_, i.e. each sentence/line in German corresponds to a sentence/line in English. The two files have the same number of lines and the German sentence on line $i$ corresponds to the English sentence on line $i$. The subtitles are extracted from the [OpenSubtitles-2018](https://opus.nlpl.eu/OpenSubtitles/corpus/version/OpenSubtitles) corpus.

Here are the first ten lines of the two files:

<style scoped>
table {
  font-size: 12px;
}
</style>
| Nb  | German (`lotr.detok.de`)         | English (`lotr.detok.en`)      |
|---|----------------------------------|--------------------------------|
| 1 | Die Welt ist im Wandel. | The world is changed.   |
| 2 | Ich spüre es im Wasser. | I feel it in the water. |
| 3 | Ich spüre es in der Erde. | I feel it in the earth. |
| 4 | Ich rieche es in der Luft. | I smell it in the air. |
| 5 | Vieles, was einst war, ist verloren, da niemand mehr lebt, der sich erinnert. | Much that once was is lost. For none now live who remember it. |
| 6 | Es begann mit dem Schmieden der Großen Ringe. | It began with the forging of the Great Rings. |
| 7 | 3 wurden den Elben gegeben, den unsterblichen, weisesten und reinsten aller Wesen. | Three were given to the Elves: Immortal, wisest and fairest of all beings. |
| 8 | 7 den Zwergenherrschern, großen Bergleuten und Handwerkern in ihren Hallen aus Stein. | Seven to the Dwarf-lords: Great miners and craftsmen of the mountain halls. |
| 9 | Und 9... 9 Ringe wurden den Menschen geschenkt, die vor allem anderen nach Macht streben. | And nine nine rings were gifted to the race of Men who, above all else, desire power. |
| 10 | Denn diese Ringe bargen die Kraft und den Willen, jedes Volk zu leiten. | For within these rings was bound the strength and will to govern each race. |


In [1]:
%pip install sentencepiece "transformers==4.55.4"


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: /fp/homes01/u01/ec-zejingw/in4080-env/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Getting started

We will use a pretrained machine translation model for German-to-English translation. The model is available on the HuggingFace model hub and can be used with the `transformers` library.

Let us first make sure that all required modules are installed:

In [2]:
# %pip install torch transformers accelerate evaluate sacrebleu sacremoses sentencepiece unbabel-comet datasets

The bilingual model is called [`opus-mt-de-en`](https://huggingface.co/Helsinki-NLP/opus-mt-de-en) and has been trained by the Helsinki-NLP group. Like (almost) all HuggingFace models, it consists of a _tokenizer_ and the _sequence-to-sequence model_ properly speaking. We need to load both separately:

In [3]:
%pip install evaluate unbabel-comet


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: /fp/homes01/u01/ec-zejingw/in4080-env/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:

%pip install sacrebleu sacremoses



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: /fp/homes01/u01/ec-zejingw/in4080-env/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "helsinki-nlp/opus-mt-de-en"
cache_dir = "/fp/projects01/ec403/hf_models"

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir = cache_dir)
translator = AutoModelForSeq2SeqLM.from_pretrained(model_name, cache_dir = cache_dir)

# Change "cuda" to "cpu" if you're running on a machine without GPU
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
translator = translator.to(device)

/fp/homes01/u01/ec-zejingw/in4080-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The `transformers` library will automatically download the models from the HuggingFace hub the first time you run this cell, so it may take a bit longer.

Let's take the first two German sentences, tokenize them, and translate them to English:

In [6]:
tokens = tokenizer(["Die Welt ist im Wandel.", "Ich spüre es im Wasser."], return_tensors="pt", padding=True)
print(tokens)

{'input_ids': tensor([[   55,   401,    29,    49,  9012,     3,     0, 58100, 58100],
        [  105,  2768,  1691,    18,    65,    49,   672,     3,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [7]:
outputs = translator.generate(**tokens.to(device), max_new_tokens=50)
print(outputs)

tensor([[58100,    36,   360,    19,  7315,     3,     0, 58100, 58100, 58100],
        [58100,    38,    85,  1595,    56,     5,     4,   616,     3,     0]],
       device='cuda:0')


__Task 2.1__ (1 point):
- What do the numbers in the `input_ids` represent?
- What is the effect of `padding=True`? How would the data look like if padding was disabled?
- What does `max_new_tokens` do? Why do you think it is important to set this parameter?

Solution:

1.
The numbers represent token IDs. Each token has a unique ID in the vocabulary.

2.
Padding makes all sequences in a batch the same length. Without padding, the sequences can have different lengths.

3.
It limits the maximum number of new tokens generated by the model. This is important to control the output length and computation time. If the value is too small, the translation may be incomplete.

We can get actual words by running the output through the `batch_decode` function of the tokenizer:

In [8]:
translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print(translations)

['The world is changing.', 'I can feel it in the water.']


__Note:__ We assume that you will run the translations from German to English. If you would like to work on the opposite translation direction (and feel comfortable evaluating the German output), you are welcome to do so. The corresponding bilingual model is called `opus-mt-en-de`.

### Data splitting

Before we move on, we need to split our data. We will evaluate different models and for that we'll need test data. We will also fine-tune a model, and for that we'll need training data. The entire Lord of the Rings dataset has 9640 lines.

__Task 2.2__ (1 point): Split the dataset in such a way that the **last** 1000 lines are used for testing and the remaining lines (8640) for training. Save the data under the following filenames: `lotr.train.de, lotr.train.en, lotr.test.de, lotr.test.en`. You can use Python code or other tools to perform the splitting.

In [9]:
# code here


with open("lotr.detok.de", encoding="utf-8") as f:
    de_lines = f.readlines()

with open("lotr.detok.en", encoding="utf-8") as f:
    en_lines = f.readlines()

assert len(de_lines) == len(en_lines) == 9640

train_de, test_de = de_lines[:-1000], de_lines[-1000:]
train_en, test_en = en_lines[:-1000], en_lines[-1000:]

for filename, lines in [
    ("lotr.train.de", train_de),
    ("lotr.train.en", train_en),
    ("lotr.test.de", test_de),
    ("lotr.test.en", test_en)
]:
    with open(filename, "w", encoding="utf-8") as f:
        f.writelines(lines)

print("Training sentences:", len(train_de))
print("Test sentences:", len(test_de))

Training sentences: 8640
Test sentences: 1000


The training sentences is 8640

The Test sentences is 1000


__Task 2.3__ (1 point): What are potential risks and drawbacks of splitting the dataset in this way? 


The dataset is not randomly split.

The last 1000 sentences may be different from the training data because they come from the end of the movie.

This may affect the evaluation results.

Now we are ready to translate the test set with our model.

__Task 2.4__ (2 points): Create a function that loads the entire `lotr.test.de` file, translates each line with the `opus-mt-de-en` model and writes its output to a new file, one sentence per line.

The easiest way to do this is to just load the entire test file into a list, tokenize and translate it, but the test set may be too large to fit on GPU memory, or it might be inefficient and slow if you use a CPU. A better alternative is to split the data into batches of 50-100 sentences and send each batch separately to the translator.

In [10]:
def translate(input_file, translation_file, tokenizer, translator, batch_size=100):
    """Translate an input file line by line using the loaded tokenizer and translator,
    and write the translations to output_file."""
    #Read the German sentences
    with open(input_file, encoding="utf-8") as f:
        sentences = [line.strip() for line in f]

    translator.eval()


    #Translate in batches
    with open(translation_file, "w", encoding="utf-8") as f:
        for i in range(0, len(sentences), batch_size):
            batch = sentences[i:i+batch_size]

            tokens = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)

            with torch.no_grad():
                outputs = translator.generate(**tokens, max_new_tokens=100)

            translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)

            for translation in translations:
                f.write(translation + "\n")

    print("Translated sentences:", len(sentences))


translate("lotr.test.de", "lotr.output_opus.en", tokenizer, translator)


Translated sentences: 1000


In [11]:

#Test translation

with open("lotr.output_opus.en", encoding="utf-8") as f:
    translations = f.readlines()

print("Total translations:", len(translations))

for sentence in translations[:5]:
    print(sentence.strip())


Total translations: 1000
There's nothing here.
Keep looking!
This stone could be anywhere.
The Arkenstein is located in these halls.
- Find him!


Conclusion

Translated 1000 German sentences into English using the outputs model

Used batches of 100 sentences!

The output file contains 1000 translations.

The first 5 sentences are in English.

Before moving on, open the output file and check that the translations look ok. In particular, the file should contain the expected number of lines and output should be in the expected language (English or German, depending on the chosen direction).

__Task 2.5__ (1 point): Open both the output file and the reference translations (`lotr.test.en` if translating from German to English) and compare the first 20 lines. How would you rate the translations of the OPUS system on a scale from 1 (incomprehensible and/or completely different meaning) to 5 (grammatically correct and meaning fully preserved)? Justify your answer.

In [12]:

#Compare the first 20 translations

with open("lotr.output_opus.en", encoding="utf-8") as f:
    translations = f.readlines()

with open("lotr.test.en", encoding="utf-8") as f:
    references = f.readlines()

for i in range(20):
    print(f"Sentence {i+1}")
    print("OPUS:", translations[i].strip())
    print("Reference:", references[i].strip())
    print()


Sentence 1
OPUS: There's nothing here.
Reference: Nothing here.

Sentence 2
OPUS: Keep looking!
Reference: Keep searching!

Sentence 3
OPUS: This stone could be anywhere.
Reference: That Jewel could be anywhere.

Sentence 4
OPUS: The Arkenstein is located in these halls.
Reference: The Arkenstone is in these halls.

Sentence 5
OPUS: - Find him!
Reference: - Find it!

Sentence 6
OPUS: You heard me.
Reference: You heard him.

Sentence 7
OPUS: - Keep looking.
Reference: - Keep looking.

Sentence 8
OPUS: All of you!
Reference: All of you!

Sentence 9
OPUS: No one rests until he's found.
Reference: No one rests until... it is found.

Sentence 10
OPUS: It's almost tempting me to leave it to you.
Reference: I am almost tempted... to let you take it.

Sentence 11
OPUS: And be it just to see how oak shield suffers.
Reference: If only... to see Oakenshield... suffer.

Sentence 12
OPUS: Seeing it destroy him.
Reference: Watch it... destroy him.

Sentence 13
OPUS: Seeing it contaminate his heart a

Compared the first 20 translations with the reference sentences.

Rated the translations 4/5.

Most translations are correct, but some have mistakes, like "Find him!" instead of "Find it!".

### Evaluation

We can now evaluate the quality of our translations. In a first step, we perform _reference-based surface-level evaluation_  using the popular BLEU score. We can do that with the `sacrebleu` module. Below is a slightly reformatted example taken from the [SacreBLEU documentation](https://github.com/mjpost/sacrebleu/tree/master?tab=readme-ov-file#using-sacrebleu-from-python):

In [13]:
from sacrebleu.metrics import BLEU

reference = ['The dog bit the man.', 'It was not unexpected.', 'The man bit him first.']
hypothesis = ['The dog bit the man.', "It wasn't surprising.", 'The man had just bitten him.']

bleu_scorer = BLEU()
# BLEU can deal with multiple references per sentence, but here we only have one, so we just enclose it in another set of brackets:
score = bleu_scorer.corpus_score(hypothesis, [reference])
print(score)

BLEU = 45.07 70.6/42.9/36.4/37.5 (BP = 1.000 ratio = 1.000 hyp_len = 17 ref_len = 17)


__Task 2.6__ (1 point): Load both the system output and the reference of your test set and compute the corpus-level BLEU score. Also compute the corpus-level chrF score. Which of the scores is higher?

In [14]:
from sacrebleu.metrics import BLEU, CHRF

def evaluate_bleu(hypothesis_file, reference_file):
    #Read translations
    with open(hypothesis_file, encoding="utf-8") as f:
        hypotheses = [line.strip() for line in f]

    #Read reference sentences
    with open(reference_file, encoding="utf-8") as f:
        references = [line.strip() for line in f]

    #Compute scores
    bleu = BLEU().corpus_score(hypotheses, [references])
    chrf = CHRF().corpus_score(hypotheses, [references])

    print("BLEU:", bleu.score)
    print("chrF:", chrf.score)

    if bleu.score > chrf.score:
        print("BLEU is higher")
    else:
        print("chrF is higher")

evaluate_bleu("lotr.output_opus.en", "lotr.test.en")

BLEU: 28.96752214356673
chrF: 51.020418298884884
chrF is higher


Conclusion


Calculated BLEU and chrF scores for the 1000 translations.

The BLEU score is 28.96752214356673 and the chrF score is 51.020418298884884

The chrF score is higher than the BLEU score.

Besides string-based metrics, neural metrics have become increasingly popular lately, since they have been shown to correlate better with human judgements. The most popular neural metric is called COMET and it can be used with the HuggingFace `evaluate` package. The example below is from the [documentation](https://huggingface.co/spaces/evaluate-metric/comet/blob/main/README.md):

In [15]:

%pip install "setuptools==80.9.0" --no-deps



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: /fp/homes01/u01/ec-zejingw/in4080-env/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:

%pip install "pyarrow==25.0.1" --no-deps



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: /fp/homes01/u01/ec-zejingw/in4080-env/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
%pip install unbabel-comet


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: /fp/homes01/u01/ec-zejingw/in4080-env/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
import evaluate

comet_metric = evaluate.load('comet')
src = ["Dem Feuer konnte Einhalt geboten werden", "Schulen und Kindergärten wurden eröffnet."]
hyp = ["The fire could be stopped", "Schools and kindergartens were open"]
ref = ["They were able to control the fire.", "Schools and kindergartens opened"]
comet_score = comet_metric.compute(predictions=hyp, references=ref, sources=src)
print(comet_score)

/fp/homes01/u01/ec-zejingw/in4080-env/lib/python3.11/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1395.87it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/fp/homes01/u01/ec-zejingw/in4080-env/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:216: Found keys that are not in the model state dict but in the che

{'mean_score': 0.9051418900489807, 'scores': [0.8385581374168396, 0.9717256426811218]}


__Task 2.7__ (1 point): Adapt this code to evaluate the output of the OPUS model. Note that COMET also requires the source text.

In [19]:
def evaluate_comet(hypothesis_file, reference_file, source_file):


    #Read translations
    
    with open(hypothesis_file, encoding="utf-8") as f:
        hypotheses = [line.strip() for line in f]

    #Read references
    
    with open(reference_file, encoding="utf-8") as f:
        references = [line.strip() for line in f]

    #Read German sentences
    
    with open(source_file, encoding="utf-8") as f:
        sources = [line.strip() for line in f]

    #Compute COMET score
    
    comet_metric = evaluate.load("comet")
    result = comet_metric.compute(predictions=hypotheses, references=references, sources=sources)

    print("COMET score:", result["mean_score"])

evaluate_comet("lotr.output_opus.en", "lotr.test.en", "lotr.test.de")


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 2128.01it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/fp/homes01/u01/ec-zejingw/in4080-env/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:216: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
/fp/homes01/u01/ec-zejingw/in4080-env/lib/python3.11/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /fp/homes01/u01/ec-zejingw/in4080-env/lib/python3.11 ...
GPU avai

COMET score: 0.8515520244240761


Conclusion

Evaluated 1000 translations using COMET.

The COMET score is 0.8515520244240761

The model received a relatively high COMET score.

### Fine-tuning

Let us see now if we can further improve the translation quality. We still haven't used the training set after all...

Fine-tuning a translation model with the `transformers` library is a bit convoluted. You need the following ingredients:
- A `Seq2SeqTrainer` object, which defines the initial model and its tokenizer, the training data, and the configuration parameters (as a `Seq2SeqTrainingArguments` object). The training process starts with the `train()` method.
- A `Seq2SeqTrainingArguments` object, which contains the configuration parameters, such as the number of training epochs, the path for saving the fine-tuned model, the learning rate etc.
- A `DataCollatorForSeq2Seq` object that takes care of splitting the training data into batches of appropriate size.
- A `DatasetDict` object containing the tokenized training data. Typically, the untokenized data is loaded into a `DatasetDict` object, and the tokenization function is applied to everything inside this `DatasetDict` using the `map()` function.

__Task 2.8__ (1 point): The code in the box below shows a working example using the pretrained OPUS model, but is limited to two sentence pairs. Complete the code to load the entire training data.

In [20]:
%pip install "accelerate==1.9.0"


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: /fp/homes01/u01/ec-zejingw/in4080-env/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import DataCollatorForSeq2Seq
from datasets import Dataset, DatasetDict

max_length = 100

#Load training data


with open("lotr.train.de", encoding="utf-8") as f:
    train_de = [line.strip() for line in f]


with open("lotr.train.en", encoding="utf-8") as f:
    train_en = [line.strip() for line in f]


#Create dataset

ds = Dataset.from_dict({
    "src_text": train_de,
    "tgt_text": train_en
})

data = DatasetDict({"train": ds})

print("Training sentences:", len(ds))



def preprocess_function(examples):
    model_inputs = tokenizer(examples["src_text"], text_target=examples["tgt_text"], max_length=max_length, truncation=True)
    return model_inputs

tokenized_datasets = data.map(preprocess_function, batched=True)
print(tokenized_datasets)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=translator)

args = Seq2SeqTrainingArguments(
    output_dir="opus-mt-de-en-lotr",
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True
)

trainer = Seq2SeqTrainer(
    translator,
    args,
    train_dataset=tokenized_datasets["train"],
    data_collator=data_collator,
    processing_class=tokenizer
)

trainer.train()

Training sentences: 8640


Map: 100%|██████████| 8640/8640 [00:01<00:00, 7560.68 examples/s]


DatasetDict({
    train: Dataset({
        features: ['src_text', 'tgt_text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 8640
    })
})


Step,Training Loss
500,1.181700


/fp/homes01/u01/ec-zejingw/in4080-env/lib/python3.11/site-packages/transformers/modeling_utils.py:3922: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[58100]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=810, training_loss=1.1135879704981675, metrics={'train_runtime': 93.912, 'train_samples_per_second': 276.003, 'train_steps_per_second': 8.625, 'total_flos': 160830792400896.0, 'train_loss': 1.1135879704981675, 'epoch': 3.0})

Conclusion

Used 8640  sentence for training.

The final training loss was 1.181700

The model was fine-tuned for three epochs, and you should have three checkpoints in the `opus-mt-de-en-lotr` directory.

__Task 2.9__ (1 point): Choose one of the checkpoints and use it to translate the test set. Evaluate the test set with BLEU, chrF and COMET. Note that locally saved model files (and tokenizers) can be loaded in the same way as models from the HuggingFace hub, e.g. with the following command: `transformers.AutoModelForSeq2SeqLM.from_pretrained("opus-mt-de-en-lotr/checkpoint-810")`

Did fine-tuning help? Did fine-tuning help? Have a look at the first rows of the files. Do you agree with the metrics?

In [22]:
# code here


#My own code



#Load fine-tuned model
from pathlib import Path
from transformers import AutoModelForSeq2SeqLM

checkpoints = sorted(Path("opus-mt-de-en-lotr").glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))

print("Checkpoints:", checkpoints)

checkpoint = str(checkpoints[-1])
fine_tuned_model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint).to(device)

#Translate test data
translate("lotr.test.de", "lotr.output_finetuned.en", tokenizer, fine_tuned_model)




#Evaluate fine-tuned model
evaluate_bleu("lotr.output_finetuned.en", "lotr.test.en")

evaluate_comet("lotr.output_finetuned.en", "lotr.test.en", "lotr.test.de")





# Compare translations


with open("lotr.output_opus.en", encoding="utf-8") as f:
    opus = f.readlines()


with open("lotr.output_finetuned.en", encoding="utf-8") as f:
    fine = f.readlines()


with open("lotr.test.en", encoding="utf-8") as f:
    references = f.readlines()


for i in range(5):
    print("Original:", opus[i].strip())
    print("Fine-tuned:", fine[i].strip())
    print("Reference:", references[i].strip())
    print()


#Thank you very much!

#I love this course!!




Checkpoints: [PosixPath('opus-mt-de-en-lotr/checkpoint-270'), PosixPath('opus-mt-de-en-lotr/checkpoint-540'), PosixPath('opus-mt-de-en-lotr/checkpoint-810')]
Translated sentences: 1000
BLEU: 33.59096926083024
chrF: 55.67500716381052
chrF is higher


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1454.13it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/fp/homes01/u01/ec-zejingw/in4080-env/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:216: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
/fp/homes01/u01/ec-zejingw/in4080-env/lib/python3.11/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /fp/homes01/u01/ec-zejingw/in4080-env/lib/python3.11 ...
GPU avai

COMET score: 0.8653928940296173
Original: There's nothing here.
Fine-tuned: There's nothing here.
Reference: Nothing here.

Original: Keep looking!
Fine-tuned: Keep looking!
Reference: Keep searching!

Original: This stone could be anywhere.
Fine-tuned: This stone could be anywhere.
Reference: That Jewel could be anywhere.

Original: The Arkenstein is located in these halls.
Fine-tuned: The Arkenstone lies in these halls.
Reference: The Arkenstone is in these halls.

Original: - Find him!
Fine-tuned: - Find him!
Reference: - Find it!



Conclusion

Tested the fine-tuned model with 1000 sentences.

The BLEU, chrF and COMET scores are higher after fine-tuning.

The translations improved, but some mistakes still remain.

For example, "Arkenstein" was corrected to "Arkenstone", but "Find him!" is still wrong.